- ROBUSTNESS AUDITING
- OOD EXTREME-ENGINE-SPECIFICATION ANALYSIS
 - MANUFACTURER-LEVEL RESIDUAL ANALYSIS

In [1]:
import sys
sys.path.append("..")

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

from joblib import load
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from joblib import load
from src.dcsr import DCSRRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [7]:
# PATHS AND SETTINGS
TRAIN_PATH = "../data/processed/S_old_temporal_train.csv" 
TEST_PATH = "../data/processed/S_new_temporal_test.csv" 
DCSR_MODEL_PATH = "../models/dcsr_model.joblib" 
XGB_MODEL_PATH = "../models/xgb_model.joblib" 
RESULTS_TABLE_DIR = "../results/tables" 
RESULTS_FIGURE_DIR = "../results/figures" 
os.makedirs(RESULTS_TABLE_DIR, exist_ok=True) 
os.makedirs(RESULTS_FIGURE_DIR, exist_ok=True) 
TARGET = "co2" 
DCSR_FEATURES = [ "engine_size", "cylinders", "engine_per_cylinder", "engine_x_cylinders", "engine_size_sq", "cylinders_sq" ] 
XGB_FEATURES = [ "make", "model", "vehicle_class", "engine_size", "cylinders", "transmission", "fuel_type", "engine_per_cylinder", "engine_x_cylinders", "engine_size_sq", "cylinders_sq" ]

In [8]:
# LOAD DATA
train_df = pd.read_csv(TRAIN_PATH) 
test_df = pd.read_csv(TEST_PATH) 
print("Temporal training data:", train_df.shape) 
print("Temporal test data:", test_df.shape)

Temporal training data: (1983, 15)
Temporal test data: (4002, 15)


In [9]:
# CHECK REQUIRED COLUMNS
required_columns = ( DCSR_FEATURES + XGB_FEATURES + [TARGET] ) 
missing_train = [ col for col in required_columns if col not in train_df.columns ] 
missing_test = [ col for col in required_columns if col not in test_df.columns ] 
if missing_train: raise ValueError( f"Missing columns in training data: {missing_train}" ) 
if missing_test: raise ValueError( f"Missing columns in test data: {missing_test}" )

In [10]:
# LOAD SAVED MODELS
dcsr_model = load(DCSR_MODEL_PATH) 
xgb_model = load(XGB_MODEL_PATH) 
print("\nDCSR model loaded.") 
print("XGBoost model loaded.")


DCSR model loaded.
XGBoost model loaded.


In [12]:
# PREPARE DATA
X_train_dcsr = train_df[DCSR_FEATURES].copy() 
X_test_dcsr = test_df[DCSR_FEATURES].copy() 
y_test = test_df[TARGET].to_numpy()

# Use training medians only
train_medians = X_train_dcsr.median() 
X_train_dcsr = X_train_dcsr.fillna(train_medians) 
X_test_dcsr = X_test_dcsr.fillna(train_medians) 
# XGBoost data 
X_test_xgb = test_df[XGB_FEATURES].copy()

In [14]:
# PREDICTIONS
dcsr_pred = dcsr_model.predict( X_test_dcsr.to_numpy() ) 
try:
    xgb_pred = xgb_model.predict(X_test_xgb)
    xgb_available = True

except Exception as e:
    xgb_available = False
    xgb_pred = None

    print("\nWARNING: XGBoost prediction failed.")
    print("Error:", e)

In [15]:
# METRIC FUNCTION
def calculate_metrics(y_true, y_pred): 

    mse = mean_squared_error(y_true, y_pred) 
    return { "R2": r2_score(y_true, y_pred), "MSE": mse, "RMSE": np.sqrt(mse), "MAE": mean_absolute_error(y_true, y_pred) }

In [16]:
# TEMPORAL TEST PERFORMANCE
dcsr_metrics = calculate_metrics( y_test, dcsr_pred ) 
print("\n" + "=" * 60) 
print("TEMPORAL TEST PERFORMANCE") 
print("=" * 60) 

print("\nDCSR") 
for metric, value in dcsr_metrics.items(): print(f"{metric}: {value:.6f}") 
if xgb_available: 
    xgb_metrics = calculate_metrics( y_test, xgb_pred ) 
    print("\nXGBoost") 
    for metric, value in xgb_metrics.items(): 
        print(f"{metric}: {value:.6f}") 

# Save temporal performance 
performance_rows = [ { "Model": "DCSR", **dcsr_metrics } ] 

if xgb_available: 
    performance_rows.append( { "Model": "XGBoost", **xgb_metrics } ) 
performance_df = pd.DataFrame(performance_rows) 

performance_path = ( f"{RESULTS_TABLE_DIR}/temporal_test_performance.csv" ) 
performance_df.to_csv( performance_path, index=False )


TEMPORAL TEST PERFORMANCE

DCSR
R2: -2.635847
MSE: 6159.660592
RMSE: 78.483505
MAE: 53.441097

XGBoost
R2: 0.971221
MSE: 48.755653
RMSE: 6.982525
MAE: 5.159612


In [23]:
# OOD THRESHOLDS
# Thresholds are calculated from the temporal training set only. 
# Engine size: 99th percentile 
# Cylinders: 95th percentile

print("=" * 70)
print("OOD THRESHOLD DIAGNOSTICS")
print("=" * 70)

print("\nTRAINING DATA")
print("-" * 70)

print(
    "Engine size:"
    f" min={train_df['engine_size'].min():.3f},"
    f" max={train_df['engine_size'].max():.3f},"
    f" 99th percentile={train_df['engine_size'].quantile(0.99):.3f}"
)

print(
    "Cylinders:"
    f" min={train_df['cylinders'].min():.0f},"
    f" max={train_df['cylinders'].max():.0f},"
    f" 95th percentile={train_df['cylinders'].quantile(0.95):.0f}"
)

print("\nTEMPORAL TEST DATA")
print("-" * 70)

print(
    "Engine size:"
    f" min={test_df['engine_size'].min():.3f},"
    f" max={test_df['engine_size'].max():.3f}"
)

print(
    "Cylinders:"
    f" min={test_df['cylinders'].min():.0f},"
    f" max={test_df['cylinders'].max():.0f}"
)

print("\nTHRESHOLDS")
print("-" * 70)

print(f"Engine threshold: {engine_threshold:.3f}")
print(f"Cylinder threshold: {cylinder_threshold:.0f}")

print("\nCOUNTS IN TEMPORAL TEST")
print("-" * 70)

print(
    "Engine >= threshold:",
    (test_df["engine_size"] >= engine_threshold).sum()
)

print(
    "Cylinders >= threshold:",
    (test_df["cylinders"] >= cylinder_threshold).sum()
)

print(
    "Either threshold:",
    (
        (test_df["engine_size"] >= engine_threshold)
        |
        (test_df["cylinders"] >= cylinder_threshold)
    ).sum()
)

OOD THRESHOLD DIAGNOSTICS

TRAINING DATA
----------------------------------------------------------------------
Engine size: min=2.500, max=8.400, 99th percentile=6.700
Cylinders: min=6, max=16, 95th percentile=12

TEMPORAL TEST DATA
----------------------------------------------------------------------
Engine size: min=0.900, max=6.200
Cylinders: min=3, max=8

THRESHOLDS
----------------------------------------------------------------------
Engine threshold: 6.700
Cylinder threshold: 12

COUNTS IN TEMPORAL TEST
----------------------------------------------------------------------
Engine >= threshold: 0
Cylinders >= threshold: 0
Either threshold: 0


In [24]:
# TRAINING DISTRIBUTION LOWER TAILS

print("=" * 70)
print("LOWER-TAIL OOD DIAGNOSTICS")
print("=" * 70)

engine_p01 = train_df["engine_size"].quantile(0.01)
engine_p05 = train_df["engine_size"].quantile(0.05)

cylinder_p01 = train_df["cylinders"].quantile(0.01)
cylinder_p05 = train_df["cylinders"].quantile(0.05)

print(f"\nEngine size:")
print(f"1st percentile: {engine_p01:.3f} L")
print(f"5th percentile: {engine_p05:.3f} L")
print(f"Test minimum:   {test_df['engine_size'].min():.3f} L")

print(f"\nCylinders:")
print(f"1st percentile: {cylinder_p01:.0f}")
print(f"5th percentile: {cylinder_p05:.0f}")
print(f"Test minimum:   {test_df['cylinders'].min():.0f}")

print("\nTEST OBSERVATIONS BELOW THRESHOLDS")
print("-" * 70)

print(
    "Engine <= 1st percentile:",
    (test_df["engine_size"] <= engine_p01).sum()
)

print(
    "Engine <= 5th percentile:",
    (test_df["engine_size"] <= engine_p05).sum()
)

print(
    "Cylinders <= 1st percentile:",
    (test_df["cylinders"] <= cylinder_p01).sum()
)

print(
    "Cylinders <= 5th percentile:",
    (test_df["cylinders"] <= cylinder_p05).sum()
)

LOWER-TAIL OOD DIAGNOSTICS

Engine size:
1st percentile: 2.700 L
5th percentile: 3.000 L
Test minimum:   0.900 L

Cylinders:
1st percentile: 6
5th percentile: 6
Test minimum:   3

TEST OBSERVATIONS BELOW THRESHOLDS
----------------------------------------------------------------------
Engine <= 1st percentile: 2742
Engine <= 5th percentile: 3343
Cylinders <= 1st percentile: 3891
Cylinders <= 5th percentile: 3891
